
# Pretrained Uni-Mol embeddings as features

*Author: Suhmi. Builds on notebooks 07 and 08.*

## Purpose

The Tg error is limited by information, not by the model (notebook 08): polymers without
similar training examples are predicted much worse. A **pretrained model** brings in
knowledge from millions of other structures.

**Uni-Mol** (DP Technology, MIT licence) was pretrained on about 19 million 3D molecular
structures. Each repeat unit is capped (`*` → H) and turned into a 512-number vector,
its **embedding** (`src/unimol_cache.py`, run in a separate environment). The embeddings are
used as extra features for XGBoost.

polyBERT, a model trained on polymers, would be a closer fit, but it is now only available
to academic institutions on request, so it is not used.

## Rules

- The same splits and folds as notebooks 04–08; cross-validation only.
- **The validation subsets are frozen.** The PCA of the embeddings uses no labels and is
  fitted only on polymers that are in no validation subset.


## 1. Setup

In [1]:

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

# Make the project's src package importable from the notebooks folder
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.polymer_features import SHAPE_COLUMNS, build_chain_features, build_features

RANDOM_STATE = 42
VALIDATION_FRACTION = 0.20
N_FOLDS = 5

target_columns = ["Tg", "FFV", "Tc", "Density", "Rg"]


## 2. Features, splits and folds

In [2]:

train = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "train.csv")

bplus_df, fingerprint_df = build_features(train["SMILES"])
chain_df = build_chain_features(train["SMILES"])
shape_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "shape_descriptors.csv",
                       index_col=0)[SHAPE_COLUMNS]
unimol_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "unimol_embeddings.csv", index_col=0)
assert shape_df.index.equals(train.index) and unimol_df.index.equals(train.index)
print("Uni-Mol embeddings:", unimol_df.shape,
      "| polymers without embedding:", int(unimol_df.isna().any(axis=1).sum()))

cv_folds = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
labels = train[target_columns].to_numpy(dtype=float)

train_rows, fold_rows, valid_rows = {}, {}, {}
for target in target_columns:
    labelled = train.index[train[target].notna()]
    train_index, valid_index = train_test_split(
        labelled, test_size=VALIDATION_FRACTION, random_state=RANDOM_STATE, shuffle=True
    )
    train_rows[target] = np.asarray(train_index)
    valid_rows[target] = np.asarray(valid_index)   # used only to exclude these polymers below
    fold_rows[target] = [train_rows[target][test_idx] for _, test_idx in cv_folds.split(train_index)]

expected_sizes = {"Tg": 408, "FFV": 5624, "Tc": 589, "Density": 490, "Rg": 491}
assert {t: len(rows) for t, rows in train_rows.items()} == expected_sizes

# PCA of the embeddings, fitted without labels on polymers outside all validation subsets
pca_rows = np.setdiff1d(train.index, np.concatenate(list(valid_rows.values())))
pca = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                    PCA(n_components=32, random_state=RANDOM_STATE))
pca.fit(unimol_df.loc[pca_rows])
unimol_pca_df = pd.DataFrame(pca.transform(unimol_df), index=train.index).add_prefix("unimol_pc")
print(f"32 principal components explain "
      f"{100 * pca[-1].explained_variance_ratio_.sum():.0f}% of the embedding variance")

descriptors_df = pd.concat([bplus_df, chain_df, shape_df], axis=1)
feature_sets = {
    "73 descriptors + D (current)": pd.concat([descriptors_df, fingerprint_df], axis=1),
    "Uni-Mol only": unimol_df,
    "73 descriptors + Uni-Mol": pd.concat([descriptors_df, unimol_df], axis=1),
    "73 descriptors + D + Uni-Mol": pd.concat([descriptors_df, fingerprint_df, unimol_df], axis=1),
    "73 descriptors + D + Uni-Mol PCA (32)": pd.concat([descriptors_df, fingerprint_df, unimol_pca_df], axis=1),
}
for name, features in feature_sets.items():
    print(f"{name}: {features.shape[1]} features")


Uni-Mol embeddings: (7973, 512) | polymers without embedding: 5


32 principal components explain 71% of the embedding variance
73 descriptors + D (current): 1097 features
Uni-Mol only: 512 features
73 descriptors + Uni-Mol: 585 features
73 descriptors + D + Uni-Mol: 1609 features
73 descriptors + D + Uni-Mol PCA (32): 1129 features



## 3. Tg: does Uni-Mol help?

Two XGBoost models on the same five folds: the tuned Tg setting (notebooks 04 and 07)
and the default setting from notebook 06. The settings are not re-tuned for the new features.


In [3]:

tg_params = dict(n_estimators=750, learning_rate=0.013339162948869679, max_depth=2,
                 min_child_weight=1, subsample=0.6550083776583973,
                 colsample_bytree=0.7394663949166917, reg_lambda=7.472397689332939)
default_params = dict(n_estimators=500, learning_rate=0.05, max_depth=4, subsample=0.8,
                      colsample_bytree=0.8)


def cross_validate(target, X, params):
    t = target_columns.index(target)
    rows_out = []
    for fold in range(N_FOLDS):
        rows = fold_rows[target][fold]
        fit_rows = np.setdiff1d(train_rows[target], rows)
        model = make_pipeline(SimpleImputer(strategy="median"),
                              XGBRegressor(**params, random_state=RANDOM_STATE, n_jobs=-1))
        y_pred = model.fit(X[fit_rows], labels[fit_rows, t]).predict(X[rows])
        y_true = labels[rows, t]
        baseline = np.mean(np.abs(y_true - np.median(labels[fit_rows, t])))
        mae = mean_absolute_error(y_true, y_pred)
        rows_out.append({"fold": fold, "MAE": mae, "error_reduction": 100 * (1 - mae / baseline),
                         "R2": r2_score(y_true, y_pred)})
    return rows_out


tg_rows = []
for set_name, features in feature_sets.items():
    X = features.to_numpy(dtype=float)
    for model_name, params in [("XGBoost (tuned)", tg_params), ("XGBoost (default)", default_params)]:
        for row in cross_validate("Tg", X, params):
            tg_rows.append({"feature_set": set_name, "model": model_name, **row})

tg_df = pd.DataFrame(tg_rows)
tg_summary = tg_df.groupby(["feature_set", "model"]).agg(
    MAE=("MAE", "mean"), error_reduction=("error_reduction", "mean"),
    error_reduction_std=("error_reduction", "std"), R2=("R2", "mean"))
display(tg_summary.loc[list(feature_sets)].round(3))


MAE  \
feature_set                           model                       
73 descriptors + D (current)          XGBoost (default)  50.596   
                                      XGBoost (tuned)    48.164   
Uni-Mol only                          XGBoost (default)  64.921   
                                      XGBoost (tuned)    64.660   
73 descriptors + Uni-Mol              XGBoost (default)  53.595   
                                      XGBoost (tuned)    52.981   
73 descriptors + D + Uni-Mol          XGBoost (default)  51.657   
                                      XGBoost (tuned)    52.134   
73 descriptors + D + Uni-Mol PCA (32) XGBoost (default)  51.214   
                                      XGBoost (tuned)    49.500   

                                                         error_reduction  \
feature_set                           model                                
73 descriptors + D (current)          XGBoost (default)           42.931   
                                      XGBoost (tuned)             45.622   
Uni-Mol only                          XGBoost (default)           26.704   
                                      XGBoost (tuned)             27.010   
73 descriptors + Uni-Mol              XGBoost (default)           39.503   
                                      XGBoost (tuned)             40.220   
73 descriptors + D + Uni-Mol          XGBoost (default)           41.705   
                                      XGBoost (tuned)             41.167   
73 descriptors + D + Uni-Mol PCA (32) XGBoost (default)           42.214   
                                      XGBoost (tuned)             44.114   

                                                         error_reduction_std  \
feature_set                           model                                    
73 descriptors + D (current)          XGBoost (default)                4.240   
                                      XGBoost (tuned)                  2.572   
Uni-Mol only                          XGBoost (default)                5.234   
                                      XGBoost (tuned)                  4.583   
73 descriptors + Uni-Mol              XGBoost (default)                3.166   
                                      XGBoost (tuned)                  3.370   
73 descriptors + D + Uni-Mol          XGBoost (default)                2.482   
                                      XGBoost (tuned)                  2.464   
73 descriptors + D + Uni-Mol PCA (32) XGBoost (default)                1.652   
                                      XGBoost (tuned)                  2.482   

                                                            R2  
feature_set                           model                     
73 descriptors + D (current)          XGBoost (default)  0.663  
                                      XGBoost (tuned)    0.687  
Uni-Mol only                          XGBoost (default)  0.407  
                                      XGBoost (tuned)    0.426  
73 descriptors + Uni-Mol              XGBoost (default)  0.619  
                                      XGBoost (tuned)    0.621  
73 descriptors + D + Uni-Mol          XGBoost (default)  0.636  
                                      XGBoost (tuned)    0.630  
73 descriptors + D + Uni-Mol PCA (32) XGBoost (default)  0.653  
                                      XGBoost (tuned)    0.670


## 4. The other properties

A quick check with default XGBoost: current features vs. current features + Uni-Mol.


In [4]:

check_sets = ["73 descriptors + D (current)", "73 descriptors + D + Uni-Mol"]
other_rows = []
for target in target_columns:
    for set_name in check_sets:
        X = feature_sets[set_name].to_numpy(dtype=float)
        for row in cross_validate(target, X, default_params):
            other_rows.append({"target": target, "feature_set": set_name, **row})

other_df = pd.DataFrame(other_rows)
other_summary = other_df.pivot_table(index="target", columns="feature_set",
                                     values="error_reduction").loc[target_columns, check_sets]
other_summary["gain (points)"] = other_summary[check_sets[1]] - other_summary[check_sets[0]]
other_summary["fold std"] = (other_df[other_df["feature_set"] == check_sets[0]]
                             .groupby("target")["error_reduction"].std())
print("Cross-validated error reduction vs. median baseline (%), default XGBoost")
display(other_summary.round(1))


Cross-validated error reduction vs. median baseline (%), default XGBoost


feature_set,73 descriptors + D (current),73 descriptors + D + Uni-Mol,gain (points),fold std
target,,,,
Tg,42.9,41.7,-1.2,4.2
FFV,67.6,65.3,-2.3,1.6
Tc,65.1,62.2,-2.9,2.5
Density,77.4,75.4,-2.1,3.3
Rg,66.0,62.6,-3.5,3.6


#### Findings: Uni-Mol embeddings

**Tg** (tuned XGBoost, 5 folds):

| Features | MAE (°C) | Error reduction |
|---|---:|---:|
| 73 descriptors + fingerprints (current) | **48.2** | **45.6%** |
| Uni-Mol only (512) | 64.7 | 27.0% |
| 73 descriptors + Uni-Mol | 53.0 | 40.2% |
| 73 descriptors + fingerprints + Uni-Mol | 52.1 | 41.2% |
| 73 descriptors + fingerprints + Uni-Mol PCA (32) | 49.5 | 44.1% |

**Other properties** (default XGBoost): adding Uni-Mol lowers the error reduction by 2–3.5 points
for every property.

- **Uni-Mol does not help; it makes every model slightly worse.** On its own it reaches only 27%
  for Tg, far below our descriptors (46%).
- The likely reasons: Uni-Mol was pretrained on small molecules, not polymers, and only sees one
  capped repeat unit; with 408 Tg polymers, 512 extra numbers mainly add noise. Compressing them
  to 32 components reduces the damage but still gives no gain.
- The chemistry-based descriptors carry more relevant information than the general-purpose
  pretrained embedding.

**Decision:** Uni-Mol embeddings are not used. The remaining option for Tg is more data.
